# Co-expression modules in breast tumours (PyWGCNA)

Builds gene co-expression modules from the TCGA-BRCA tumour expression
matrix, and exports an edge list for network visualisation.

**Produces** the module assignment and the thresholded edge list in
`../../results/tables/wgcna/`. The network figure itself (Supplementary
Figure S3) is laid out in Cytoscape from that edge list — see
`02_compound_gene_networks.ipynb`.

⚠️ Heavy: the VST normalisation and the module search run on the full
expression matrix, and the saved PyWGCNA object is several GB. Intermediate
files go to `../../data/external/wgcna/`, which is not tracked.

In [ ]:
import anndata
import numpy as np
import pandas as pd
import PyWGCNA
from pydeseq2.dds import DeseqDataSet

TABLES = "../../results/tables/wgcna"
DATA = "../../data/external"
WORK = f"{DATA}/wgcna"          # large intermediates, not tracked

TOM_THRESHOLD = 0.13            # topological overlap above which two genes are linked

## 1. Expression matrix and sample annotation

In [ ]:
expression = pd.read_csv(
    f"{DATA}/TCGA/DGEA_Protocol/ExpressionProfiles_GTEX-TCGA.csv", index_col=0
)

# Samples as rows, genes as columns; the first two columns are gene identifiers.
samples = expression.T
samples.columns = samples.iloc[0]
samples = samples[2:]
counts = np.round(samples).astype(int)

sample_info = pd.DataFrame(index=counts.index)
sample_info["sample_type"] = np.where(
    sample_info.index.str.startswith("TCGA"), "Tumor", "Healthy"
)
print(sample_info["sample_type"].value_counts().to_string())

In [ ]:
adata = anndata.AnnData(X=counts, obs=sample_info)
adata.var_names_make_unique()
adata_tumor = adata[adata.obs["sample_type"] == "Tumor"].copy()

## 2. Variance-stabilising transformation

WGCNA correlates expression across samples, so it needs a variance-stabilised
matrix rather than raw counts: without it, the correlation structure is
dominated by the mean-variance relationship of count data.

Modules are built on the **tumour** samples only. Including the normal
samples would create modules that merely separate tumour from normal, which
is already the subject of `analyses/04`.

In [ ]:
dds = DeseqDataSet(adata=adata_tumor, design_factors="sample_type", quiet=False)
dds.deseq2()
dds.vst()

wgcna_input = pd.DataFrame(
    dds.layers["vst_counts"], index=adata_tumor.obs_names, columns=adata_tumor.var_names
)
wgcna_input.to_csv(f"{WORK}/WGCNA_input_Tumor.csv")
print(f"{wgcna_input.shape[0]} tumours x {wgcna_input.shape[1]} genes")

## 3. Gene and sample metadata for PyWGCNA

In [ ]:
ensembl = pd.read_csv(
    f"{DATA}/TCGA/DGEA_Protocol/probeMap_gencode.v23.annotation.gene.probemap", sep="\t"
)[["id", "gene"]].rename(columns={"id": "ensembl_id", "gene": "gene_name"})

coding = pd.read_csv(f"{DATA}/TCGA/DGEA_Protocol/zz_gene.protein.coding.csv")[
    ["Gene_ID", "Gene_Symbol", "Gene_Type"]
].rename(columns={"Gene_ID": "gene_id", "Gene_Symbol": "gene_name",
                  "Gene_Type": "gene_type"})

gene_metadata = ensembl.merge(coding, on="gene_name", how="left").dropna()
gene_metadata = gene_metadata[gene_metadata["ensembl_id"].isin(wgcna_input.columns)]
gene_metadata.to_csv(f"{TABLES}/TCGA_gene_metadata.csv", index=False)

sample_metadata = sample_info.reset_index(names="Sample_id")
sample_metadata = sample_metadata[sample_metadata["Sample_id"].str.startswith("TCGA")]
sample_metadata.to_csv(f"{TABLES}/TCGA_metadata_samples.csv", index=False)

## 4. Module detection

In [ ]:
wgcna = PyWGCNA.WGCNA(
    name="TCGA_WGCNA_tumor",
    species="homo_sapiens",
    geneExpPath=f"{WORK}/WGCNA_input_Tumor.csv",
    outputPath=f"{WORK}/",
    save=True,
)
wgcna.preprocess()
wgcna.findModules()
wgcna.updateSampleInfo(path=f"{TABLES}/TCGA_metadata_samples.csv", sep=",")
wgcna.updateGeneInfo(path=f"{TABLES}/TCGA_gene_metadata.csv", sep=",")
wgcna.saveWGCNA()

In [ ]:
# To resume from the saved object instead of recomputing:
# wgcna = PyWGCNA.readWGCNA(f"{WORK}/TCGA_WGCNA_tumor.p")

module_sizes = (
    wgcna.datExpr.var["moduleColors"].value_counts().rename("genes").to_frame()
)
print(f"{len(module_sizes)} modules")
module_sizes.head(10)

In [ ]:
wgcna.datExpr.var.to_csv(f"{TABLES}/pyWGCNA_TCGA_datExpr_var_tumor.csv")

## 5. Edge list for Cytoscape

The topological overlap matrix is thresholded into a binary adjacency matrix,
then flattened into the edge list that Supplementary Figure S3 is drawn from.
The threshold is a layout choice, not a statistical one: it controls how dense
the drawn network is.

In [ ]:
tom = wgcna.TOM
ensembl_to_symbol = dict(zip(gene_metadata["ensembl_id"], gene_metadata["gene_name"]))
tom.columns = [ensembl_to_symbol[g] for g in tom.columns]
tom.index = [ensembl_to_symbol[g] for g in tom.index]

adjacency = (tom > TOM_THRESHOLD).astype(int)
np.fill_diagonal(adjacency.values, 0)          # no self-loops

edges = adjacency.stack().reset_index()
edges.columns = ["Gene1", "Gene2", "Connected"]
edges = edges[edges["Connected"] == 1][["Gene1", "Gene2"]]

threshold_tag = str(TOM_THRESHOLD).replace(".", "_")
edges.to_csv(f"{TABLES}/TCGA_edge_list_threshold_{threshold_tag}_tumor.csv", index=False)
print(f"{len(edges)} edges above TOM > {TOM_THRESHOLD}")